# 🧥 FitLoop — Phase 1: Dataset Exploration & Acquisition Verification

**Dataset:** DeepFashion2  
**Notebook:** `01_dataset_exploration.ipynb`  
**Goal:** Thoroughly audit the DeepFashion2 dataset to understand its structure, quality, diversity and annotation completeness — without modifying any original files.

---

## 📋 Stages
| # | Stage | Purpose |
|---|-------|---------|
| 1 | Environment & Path Configuration | Set up paths, detect splits |
| 2 | Dataset Inventory | Count files, verify correspondence |
| 3 | Annotation Exploration | Parse JSONs, analyse categories, boxes, masks |
| 4 | Image Quality Analysis | Dimensions, blur, brightness, corruption |
| 5 | Dataset Diversity | Class balance, viewpoints, occlusion |
| 6 | Dataset Selection Plan | Subset strategy (15–25k images) |
| 7 | Dataset Split Design | Train / Val / Test split proposal |

> ⚠️ **Rules:** No model training · No dataset downloads · No file deletions or modifications

---
## Stage 1 — Environment & Path Configuration

We start by importing only the libraries we actually need and discovering what dataset splits exist on disk.  
We do **not** hard-code an assumed folder structure — we detect it at runtime.

In [ ]:
# ── Standard library ────────────────────────────────────────────────────────
import os
import json
import random
import hashlib
import warnings
import collections
from pathlib import Path
from datetime import datetime

# ── Third-party ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patches as patches
from matplotlib.colors import to_rgba
import seaborn as sns
from PIL import Image, ImageStat
import cv2
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100
random.seed(42)
np.random.seed(42)

print('✅ Libraries loaded successfully.')
print(f'   NumPy  : {np.__version__}')
print(f'   Pandas : {pd.__version__}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  ⚙️  CENTRALIZED CONFIGURATION & DATASET PATHS
#     Imports paths directly from wardrobe_ml/config.py
# ════════════════════════════════════════════════════════════════════════════
import sys
from pathlib import Path

# ── Robust project-root discovery ───────────────────────────────────────────
# Search cwd and its ancestor directories to locate wardrobe_ml/config.py
def find_ml_root() -> Path:
    cwd = Path.cwd().resolve()
    for parent in [cwd] + list(cwd.parents):
        if (parent / 'config.py').is_file():
            return parent
        if (parent / 'wardrobe_ml' / 'config.py').is_file():
            return (parent / 'wardrobe_ml').resolve()
    raise FileNotFoundError(f'Could not locate wardrobe_ml/config.py starting from {cwd}')

ML_DIR = find_ml_root()
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

# ── Import centralized variables from config.py ─────────────────────────────
from config import (
    PROJECT_ROOT,
    DATASET_ROOT,
    TRAIN_IMAGE_DIR,
    TRAIN_ANNOTATION_DIR,
    VALIDATION_IMAGE_DIR,
    VALIDATION_ANNOTATION_DIR,
    TEST_IMAGE_DIR,
    TEST_METADATA_DIR,
    TEST_HAS_GROUND_TRUTH,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    OUTPUT_DIR,
    PROJECT_METADATA_DIR,
    cfg,
)

# Metadata & audit report paths
METADATA_DIR = PROJECT_METADATA_DIR
METADATA_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_REPORT = METADATA_DIR / 'dataset_audit_report.json'

# ── Verification output ─────────────────────────────────────────────────────
print('✅ Centralized configuration loaded successfully.')
print(f'   Config source       : {ML_DIR / "config.py"}')
print(f'   DATASET_ROOT        : {DATASET_ROOT} (exists: {DATASET_ROOT.exists()})')
print(f'   TRAIN_IMAGE_DIR     : {TRAIN_IMAGE_DIR} (exists: {TRAIN_IMAGE_DIR.exists()})')
print(f'   TRAIN_ANNO_DIR      : {TRAIN_ANNOTATION_DIR} (exists: {TRAIN_ANNOTATION_DIR.exists()})')
print(f'   VAL_IMAGE_DIR       : {VALIDATION_IMAGE_DIR} (exists: {VALIDATION_IMAGE_DIR.exists()})')
print(f'   VAL_ANNO_DIR        : {VALIDATION_ANNOTATION_DIR} (exists: {VALIDATION_ANNOTATION_DIR.exists()})')
print(f'   TEST_IMAGE_DIR      : {TEST_IMAGE_DIR} (exists: {TEST_IMAGE_DIR.exists()})')
print(f'   TEST_METADATA_DIR   : {TEST_METADATA_DIR} (exists: {TEST_METADATA_DIR.exists()})')
print(f'   TEST_HAS_GT         : {TEST_HAS_GROUND_TRUTH} (ground truth withheld in benchmark)')
print(f'   METADATA_DIR        : {METADATA_DIR}')
print(f'   AUDIT_REPORT        : {AUDIT_REPORT}')


In [ ]:
# ── Auto-detect splits and their image / annotation sub-paths ───────────────
#
# DeepFashion2 may be organised as:
#   <root>/train/image/  + <root>/train/annos/
#   <root>/validation/image/ + <root>/validation/annos/
#   <root>/test/image/  (no public annotations)
#
# We detect whatever actually exists rather than assuming.

IMAGE_SUBDIR_NAMES = ['image', 'images', 'imgs', 'JPEGImages', 'photos']
ANNO_SUBDIR_NAMES  = ['annos', 'annotations', 'labels', 'json']

def find_subdir(parent: Path, candidates: list) -> Path | None:
    """Return the first existing subdirectory from a list of candidate names."""
    for name in candidates:
        p = parent / name
        if p.is_dir():
            return p
    return None

splits_found = {}

# Walk one level deep to find candidate split folders
top_level_dirs = [d for d in DATASET_ROOT.iterdir() if d.is_dir()]
print(f'📂 Top-level directories inside dataset root:')
for d in sorted(top_level_dirs):
    print(f'   {d.name}/')

print()
KNOWN_SPLITS = ['train', 'validation', 'val', 'test']

for split_dir in sorted(top_level_dirs):
    split_name = split_dir.name.lower()
    # Normalise val → validation
    if split_name in KNOWN_SPLITS or any(s in split_name for s in KNOWN_SPLITS):
        img_dir  = find_subdir(split_dir, IMAGE_SUBDIR_NAMES)
        anno_dir = find_subdir(split_dir, ANNO_SUBDIR_NAMES)
        splits_found[split_dir.name] = {
            'root'    : split_dir,
            'img_dir' : img_dir,
            'anno_dir': anno_dir,
        }
        status_img  = '✅' if img_dir  else '❌ (not found)'
        status_anno = '✅' if anno_dir else '❌ (not found)'
        print(f'  Split  : {split_dir.name}')
        print(f'    Images      : {status_img}  → {img_dir}')
        print(f'    Annotations : {status_anno} → {anno_dir}')
        print()

if not splits_found:
    print('⚠️  No recognised splits detected. Check DATASET_ROOT or folder names.')
else:
    print(f'🔍 Detected {len(splits_found)} split(s): {list(splits_found.keys())}')

---
## Stage 2 — Dataset Inventory

Count every image and annotation file, verify image↔annotation correspondence, and flag missing or unexpected files.

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tiff', '.tif'}

def collect_files(directory: Path, extensions: set) -> list[Path]:
    """Recursively collect all files matching given extensions."""
    if directory is None or not directory.exists():
        return []
    return sorted([f for f in directory.rglob('*') if f.suffix.lower() in extensions])

inventory = {}   # will hold per-split counts

for split_name, paths in splits_found.items():
    img_dir  = paths['img_dir']
    anno_dir = paths['anno_dir']

    images      = collect_files(img_dir,  IMAGE_EXTENSIONS)
    annotations = collect_files(anno_dir, {'.json'})

    # Format distribution
    fmt_counts = collections.Counter(f.suffix.lower() for f in images)

    # Correspondence check
    # DeepFashion2: image 000001.jpg ↔ anno 000001.json (same stem)
    img_stems  = {f.stem for f in images}
    anno_stems = {f.stem for f in annotations}

    imgs_missing_anno  = img_stems  - anno_stems   # images without annotation
    annos_missing_img  = anno_stems - img_stems    # annotations without image
    matched_pairs      = img_stems  & anno_stems   # correctly paired

    inventory[split_name] = {
        'image_count'       : len(images),
        'anno_count'        : len(annotations),
        'matched_pairs'     : len(matched_pairs),
        'imgs_missing_anno' : sorted(list(imgs_missing_anno))[:20],  # cap at 20 for display
        'annos_missing_img' : sorted(list(annos_missing_img))[:20],
        'format_counts'     : dict(fmt_counts),
        'image_paths'       : images,
        'anno_paths'        : annotations,
    }

    print(f'╔══ Split: {split_name.upper()} ══')
    print(f'║  Total images      : {len(images):>8,}')
    print(f'║  Total annotations : {len(annotations):>8,}')
    print(f'║  Matched pairs     : {len(matched_pairs):>8,}')
    print(f'║  Imgs w/o anno     : {len(imgs_missing_anno):>8,}',
          '  ← these are unannotated (may be test-set normal)' if imgs_missing_anno else '')
    print(f'║  Annos w/o img     : {len(annos_missing_img):>8,}')
    print(f'║  Image formats     : {dict(fmt_counts)}')
    if imgs_missing_anno:
        print(f'║  Sample missing: {list(imgs_missing_anno)[:5]}')
    print('╚' + '═'*40)

In [ ]:
# ── Summary table ────────────────────────────────────────────────────────────
summary_rows = []
for split_name, inv in inventory.items():
    summary_rows.append({
        'Split'              : split_name,
        'Images'             : inv['image_count'],
        'Annotations'        : inv['anno_count'],
        'Matched Pairs'      : inv['matched_pairs'],
        'Imgs Missing Anno'  : len(inv['imgs_missing_anno']),
        'Annos Missing Img'  : len(inv['annos_missing_img']),
    })

df_summary = pd.DataFrame(summary_rows)
total_row  = pd.DataFrame([{
    'Split': 'TOTAL',
    'Images': df_summary['Images'].sum(),
    'Annotations': df_summary['Annotations'].sum(),
    'Matched Pairs': df_summary['Matched Pairs'].sum(),
    'Imgs Missing Anno': df_summary['Imgs Missing Anno'].sum(),
    'Annos Missing Img': df_summary['Annos Missing Img'].sum(),
}])
df_summary = pd.concat([df_summary, total_row], ignore_index=True)

print('\n📊 Inventory Summary:')
print(df_summary.to_string(index=False))

In [ ]:
# ── Visualise split sizes ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart – image counts per split
split_names  = [r['Split'] for r in summary_rows]
image_counts = [r['Images'] for r in summary_rows]
anno_counts  = [r['Annotations'] for r in summary_rows]

x = np.arange(len(split_names))
width = 0.35
axes[0].bar(x - width/2, image_counts, width, label='Images',      color='steelblue',  alpha=0.85)
axes[0].bar(x + width/2, anno_counts,  width, label='Annotations', color='darkorange', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(split_names)
axes[0].set_title('Images vs Annotations per Split')
axes[0].set_ylabel('Count')
axes[0].legend()
for i, (ic, ac) in enumerate(zip(image_counts, anno_counts)):
    axes[0].text(i - width/2, ic + 50, f'{ic:,}', ha='center', va='bottom', fontsize=8)
    axes[0].text(i + width/2, ac + 50, f'{ac:,}', ha='center', va='bottom', fontsize=8)

# Pie chart – proportion of images per split
axes[1].pie(image_counts, labels=split_names, autopct='%1.1f%%',
            colors=['steelblue', 'mediumseagreen', 'salmon'],
            startangle=140, pctdistance=0.75)
axes[1].set_title('Image Distribution Across Splits')

plt.suptitle('DeepFashion2 — Dataset Split Overview', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Stage 3 — Annotation Exploration

DeepFashion2 annotations follow a unique per-image JSON format.  
Each JSON has a `source` key plus item keys `item1`, `item2`, … for each garment in the image.  
Each item contains: `category_id`, `category_name`, `bounding_box`, `segmentation`, `landmarks`, `occlusion`, `zoom_in`, `viewpoint`.

In [ ]:
# ── DeepFashion2 category map ────────────────────────────────────────────────
CATEGORY_MAP = {
    1 : 'short sleeve top',
    2 : 'long sleeve top',
    3 : 'short sleeve outwear',
    4 : 'long sleeve outwear',
    5 : 'vest',
    6 : 'sling',
    7 : 'shorts',
    8 : 'trousers',
    9 : 'skirt',
    10: 'short sleeve dress',
    11: 'long sleeve dress',
    12: 'vest dress',
    13: 'sling dress',
}

print('📋 DeepFashion2 Garment Categories:')
for cid, name in CATEGORY_MAP.items():
    print(f'   [{cid:>2}] {name}')

In [ ]:
def parse_annotation(json_path: Path) -> dict:
    """
    Parse a single DeepFashion2 annotation JSON.
    Returns a dict with 'source' and a list of garment 'items'.
    """
    with open(json_path, 'r') as f:
        raw = json.load(f)

    source = raw.get('source', 'unknown')
    items  = []
    for key, val in raw.items():
        if key.startswith('item'):
            items.append(val)

    return {'source': source, 'items': items, 'num_items': len(items)}


# ── Inspect a few sample annotations from the train split ───────────────────
SAMPLE_SPLIT = 'train'   # change if train is not present
if SAMPLE_SPLIT not in inventory:
    SAMPLE_SPLIT = list(inventory.keys())[0]

sample_annos = inventory[SAMPLE_SPLIT]['anno_paths'][:5]

print(f'🔍 Inspecting 5 sample annotations from split: [{SAMPLE_SPLIT}]\n')
for anno_path in sample_annos:
    parsed = parse_annotation(anno_path)
    print(f'  File   : {anno_path.name}')
    print(f'  Source : {parsed["source"]}')
    print(f'  Items  : {parsed["num_items"]} garment(s)')
    for i, item in enumerate(parsed['items']):
        cat_id  = item.get('category_id', '?')
        cat_nm  = CATEGORY_MAP.get(cat_id, item.get('category_name', '?'))
        bbox    = item.get('bounding_box', [])
        seg_len = len(item.get('segmentation', []))
        lms     = len(item.get('landmarks', []))
        occ     = item.get('occlusion', '?')
        zoom    = item.get('zoom_in', '?')
        view    = item.get('viewpoint', '?')
        print(f'    item{i+1}: [{cat_id:>2}] {cat_nm:<22} '
              f'bbox={bbox}  seg_pts={seg_len}  landmarks={lms}  '
              f'occ={occ}  zoom={zoom}  view={view}')
    print()

In [ ]:
# ── Full annotation scan for the train split ─────────────────────────────────
#    We iterate all annotation files and collect key statistics.
#    This may take a few minutes for large splits.

SCAN_SPLIT   = 'train'
if SCAN_SPLIT not in inventory:
    SCAN_SPLIT = list(inventory.keys())[0]

anno_paths = inventory[SCAN_SPLIT]['anno_paths']

records = []   # one row per garment item

for anno_path in tqdm(anno_paths, desc=f'Scanning {SCAN_SPLIT} annotations'):
    try:
        parsed = parse_annotation(anno_path)
        for item in parsed['items']:
            bbox = item.get('bounding_box', [])
            seg  = item.get('segmentation', [])
            lms  = item.get('landmarks', [])
            records.append({
                'file'           : anno_path.stem,
                'source'         : parsed['source'],
                'category_id'    : item.get('category_id'),
                'category_name'  : CATEGORY_MAP.get(item.get('category_id'),
                                                    item.get('category_name', 'unknown')),
                'occlusion'      : item.get('occlusion'),
                'zoom_in'        : item.get('zoom_in'),
                'viewpoint'      : item.get('viewpoint'),
                'has_bbox'       : len(bbox) == 4,
                'bbox_w'         : (bbox[2] - bbox[0]) if len(bbox) == 4 else None,
                'bbox_h'         : (bbox[3] - bbox[1]) if len(bbox) == 4 else None,
                'bbox_area'      : ((bbox[2]-bbox[0]) * (bbox[3]-bbox[1]))
                                    if len(bbox) == 4 else None,
                'has_seg'        : len(seg) > 0,
                'seg_point_count': len(seg),
                'has_landmarks'  : len(lms) > 0,
                'landmark_count' : len(lms) // 3 if len(lms) > 0 else 0,
                'num_items_in_img': parsed['num_items'],
            })
    except Exception as e:
        records.append({'file': anno_path.stem, 'source': 'ERROR', 'error': str(e)})

df = pd.DataFrame(records)
print(f'\n✅ Scanned {len(anno_paths):,} annotation files  →  {len(df):,} garment records.')
print(f'\nDataFrame shape: {df.shape}')
df.head()

In [ ]:
# ── Category distribution ─────────────────────────────────────────────────────
cat_counts = (
    df.groupby(['category_id', 'category_name'])
      .size()
      .reset_index(name='count')
      .sort_values('count', ascending=False)
)
cat_counts['pct'] = (cat_counts['count'] / cat_counts['count'].sum() * 100).round(2)

print('📊 Garment Category Distribution (Training Split):')
print(cat_counts.to_string(index=False))

plt.figure(figsize=(12, 5))
palette = sns.color_palette('tab10', n_colors=len(cat_counts))
bars = plt.barh(cat_counts['category_name'], cat_counts['count'],
                color=palette, edgecolor='white')
plt.xlabel('Number of Annotated Instances')
plt.title('DeepFashion2 — Garment Category Distribution (Train Split)', fontweight='bold')
for bar, count in zip(bars, cat_counts['count']):
    plt.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
             f'{count:,}', va='center', fontsize=9)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# ── Bounding box analysis ─────────────────────────────────────────────────────
df_bbox = df[df['has_bbox'] == True].copy()

print(f'✅ {len(df_bbox):,} items have valid bounding boxes.')
print(f'⚠️  {len(df) - len(df_bbox):,} items are missing bounding boxes.')
print()
print('Bounding Box Statistics (px):')
print(df_bbox[['bbox_w', 'bbox_h', 'bbox_area']].describe().round(1).to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df_bbox['bbox_w'].dropna(),   bins=60, color='steelblue',    alpha=0.8, edgecolor='white')
axes[0].set_title('BBox Width Distribution');  axes[0].set_xlabel('Width (px)')
axes[1].hist(df_bbox['bbox_h'].dropna(),   bins=60, color='mediumseagreen', alpha=0.8, edgecolor='white')
axes[1].set_title('BBox Height Distribution'); axes[1].set_xlabel('Height (px)')
axes[2].hist(df_bbox['bbox_area'].dropna(), bins=60, color='darkorange',   alpha=0.8, edgecolor='white')
axes[2].set_title('BBox Area Distribution');   axes[2].set_xlabel('Area (px²)')
plt.suptitle('Bounding Box Dimensions', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Segmentation & landmark coverage ─────────────────────────────────────────
print('Segmentation Coverage:')
print(f'  Items with segmentation polygons : {df["has_seg"].sum():,}  '
      f'({df["has_seg"].mean()*100:.1f}%)')
print(f'  Items with landmark annotations  : {df["has_landmarks"].sum():,}  '
      f'({df["has_landmarks"].mean()*100:.1f}%)')
print()
print('Avg segmentation polygon points per item:')
print(f'  {df["seg_point_count"].mean():.1f}  (min={df["seg_point_count"].min()}, max={df["seg_point_count"].max()})')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
seg_labels  = ['Has Segmentation', 'No Segmentation']
seg_sizes   = [df['has_seg'].sum(), (~df['has_seg']).sum()]
lm_labels   = ['Has Landmarks', 'No Landmarks']
lm_sizes    = [df['has_landmarks'].sum(), (~df['has_landmarks']).sum()]

axes[0].pie(seg_sizes, labels=seg_labels, autopct='%1.1f%%',
            colors=['#4C72B0', '#DD8452'], startangle=90)
axes[0].set_title('Segmentation Annotation Coverage')
axes[1].pie(lm_sizes, labels=lm_labels, autopct='%1.1f%%',
            colors=['#55A868', '#C44E52'], startangle=90)
axes[1].set_title('Landmark Annotation Coverage')
plt.suptitle('Annotation Completeness', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualise annotated samples ───────────────────────────────────────────────
#    Draw bounding boxes and segmentation polygons on a sample grid.

CMAP = plt.cm.get_cmap('tab10', 13)

def draw_annotations(img_path: Path, anno_path: Path, ax: plt.Axes) -> None:
    """Overlay bounding boxes and segmentation masks on an image."""
    try:
        img = Image.open(img_path).convert('RGB')
        ax.imshow(img)
    except Exception:
        ax.text(0.5, 0.5, 'Image load failed', ha='center', va='center')
        ax.axis('off')
        return

    if anno_path and anno_path.exists():
        parsed = parse_annotation(anno_path)
        for item in parsed['items']:
            cat_id  = item.get('category_id', 0)
            cat_nm  = CATEGORY_MAP.get(cat_id, 'unknown')
            bbox    = item.get('bounding_box', [])
            seg     = item.get('segmentation', [])
            color   = CMAP((cat_id - 1) % 13)

            # Bounding box
            if len(bbox) == 4:
                x1, y1, x2, y2 = bbox
                rect = patches.Rectangle(
                    (x1, y1), x2 - x1, y2 - y1,
                    linewidth=2, edgecolor=color, facecolor='none'
                )
                ax.add_patch(rect)
                ax.text(x1, y1 - 4, cat_nm, color=color, fontsize=6,
                        bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.6, ec='none'))

            # Segmentation polygon
            if len(seg) >= 6:
                seg_arr = np.array(seg).reshape(-1, 2)
                poly = plt.Polygon(seg_arr, closed=True,
                                   fill=True, facecolor=(*color[:3], 0.25),
                                   edgecolor=color, linewidth=1)
                ax.add_patch(poly)

    ax.axis('off')
    ax.set_title(img_path.stem, fontsize=7)


# Sample 9 matched pairs
img_dir  = inventory[SCAN_SPLIT]['img_dir']
anno_dir = inventory[SCAN_SPLIT]['anno_dir']

matched_stems = [
    s for s in
    {f.stem for f in inventory[SCAN_SPLIT]['image_paths']} &
    {f.stem for f in inventory[SCAN_SPLIT]['anno_paths']}
]
sample_stems = random.sample(matched_stems, min(9, len(matched_stems)))

fig, axes = plt.subplots(3, 3, figsize=(15, 13))
for ax, stem in zip(axes.flat, sample_stems):
    # Find image (try common extensions)
    img_path = None
    for ext in IMAGE_EXTENSIONS:
        candidate = img_dir / (stem + ext)
        if candidate.exists():
            img_path = candidate
            break
    anno_path = anno_dir / (stem + '.json')
    if img_path:
        draw_annotations(img_path, anno_path, ax)
    else:
        ax.text(0.5, 0.5, f'Image not found\n{stem}', ha='center', va='center')
        ax.axis('off')

plt.suptitle('Sample Annotated Images — Bounding Boxes & Segmentation Masks',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Stage 4 — Image Quality Analysis

We analyse image dimensions, aspect ratios, file sizes, blur, brightness and contrast.  
Corrupted images are flagged but **never deleted automatically**.

In [ ]:
def compute_blur_score(img_array: np.ndarray) -> float:
    """
    Laplacian variance — higher = sharper.
    Values below ~100 are typically blurry.
    """
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def compute_brightness_contrast(img_array: np.ndarray):
    """Mean brightness and std-dev (contrast proxy) on grayscale."""
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    return float(gray.mean()), float(gray.std())


# ── Sample up to 3,000 images for quality analysis (full scan can be slow) ───
MAX_QUALITY_SAMPLE = 3000
img_paths_all = inventory[SCAN_SPLIT]['image_paths']
sample_img_paths = random.sample(
    img_paths_all, min(MAX_QUALITY_SAMPLE, len(img_paths_all))
)

print(f'🔎 Analysing {len(sample_img_paths):,} sampled images for quality metrics...')
print('   (Subsample used for speed; full scan can be run by setting MAX_QUALITY_SAMPLE to a larger value)')

quality_records = []
corrupted_files = []

for img_path in tqdm(sample_img_paths, desc='Quality scan'):
    try:
        with Image.open(img_path) as pil_img:
            w, h    = pil_img.size
            fmt     = pil_img.format
            mode    = pil_img.mode
            # Force full decode to catch corrupt JPEGs
            pil_img.verify()

        # Re-open after verify (verify closes the file)
        with Image.open(img_path) as pil_img:
            img_arr = np.array(pil_img.convert('RGB'))

        blur       = compute_blur_score(img_arr)
        brightness, contrast = compute_brightness_contrast(img_arr)
        file_size  = img_path.stat().st_size
        aspect     = round(w / h, 3)

        quality_records.append({
            'file'       : img_path.stem,
            'width'      : w,
            'height'     : h,
            'aspect'     : aspect,
            'megapixels' : round(w * h / 1e6, 3),
            'file_size_kb': round(file_size / 1024, 1),
            'blur_score' : round(blur, 2),
            'brightness' : round(brightness, 2),
            'contrast'   : round(contrast, 2),
            'mode'       : mode,
            'format'     : fmt,
        })
    except Exception as e:
        corrupted_files.append({'file': str(img_path), 'error': str(e)})

df_quality = pd.DataFrame(quality_records)
print(f'\n✅ Quality scan complete.')
print(f'   Valid images   : {len(df_quality):,}')
print(f'   Corrupted / unreadable : {len(corrupted_files):,}')
if corrupted_files:
    print('\n⚠️  Corrupted files (first 10):')
    for c in corrupted_files[:10]:
        print(f'   {c["file"]}  →  {c["error"]}')

In [ ]:
print('Image Quality Statistics:')
print(df_quality[['width', 'height', 'megapixels', 'file_size_kb',
                   'blur_score', 'brightness', 'contrast', 'aspect']]
      .describe().round(2).to_string())

# ── Blurry image threshold ────────────────────────────────────────────────────
BLUR_THRESHOLD = 100.0
blurry = df_quality[df_quality['blur_score'] < BLUR_THRESHOLD]
print(f'\n📌 Blurry images (blur_score < {BLUR_THRESHOLD}): '
      f'{len(blurry):,}  ({len(blurry)/len(df_quality)*100:.1f}%)')

In [ ]:
# ── Quality distributions ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

metrics = [
    ('width',         'Image Width (px)',         'steelblue'),
    ('height',        'Image Height (px)',        'mediumseagreen'),
    ('aspect',        'Aspect Ratio (W/H)',       'darkorange'),
    ('file_size_kb',  'File Size (KB)',           'mediumpurple'),
    ('blur_score',    'Blur Score (↑ = sharper)', 'crimson'),
    ('brightness',    'Brightness (0–255)',       'goldenrod'),
]

for ax, (col, title, color) in zip(axes.flat, metrics):
    data = df_quality[col].dropna()
    # Clip extreme outliers for readability
    p1, p99 = data.quantile(0.01), data.quantile(0.99)
    ax.hist(data.clip(p1, p99), bins=60, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.axvline(data.median(), color='black', linestyle='--', linewidth=1.2,
               label=f'Median: {data.median():.1f}')
    ax.legend(fontsize=8)

plt.suptitle('Image Quality Metric Distributions (Sampled)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter: Brightness vs Contrast, coloured by blur score ──────────────────
plt.figure(figsize=(9, 6))
sc = plt.scatter(
    df_quality['brightness'], df_quality['contrast'],
    c=df_quality['blur_score'].clip(0, 500),
    cmap='RdYlGn', alpha=0.4, s=5
)
plt.colorbar(sc, label='Blur Score (clipped at 500)')
plt.xlabel('Brightness (mean grayscale)')
plt.ylabel('Contrast (std grayscale)')
plt.title('Brightness vs Contrast (colour = sharpness)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Stage 5 — Dataset Diversity

Measure class imbalance, viewpoint variety, occlusion levels, multi-garment complexity and image source diversity.

In [ ]:
# ── Viewpoint distribution ────────────────────────────────────────────────────
# DeepFashion2 viewpoint codes: 1=no_flip, 2=flip
VIEWPOINT_MAP = {1: 'No Flip', 2: 'Flip'}

viewpoint_counts = df['viewpoint'].value_counts().sort_index()
print('👁️  Viewpoint Distribution:')
for vp, cnt in viewpoint_counts.items():
    print(f'   [{vp}] {VIEWPOINT_MAP.get(vp, str(vp)):12s}: {cnt:>8,}  ({cnt/len(df)*100:.1f}%)')

In [ ]:
# ── Occlusion distribution ────────────────────────────────────────────────────
# 1=slight, 2=medium, 3=heavy
OCCLUSION_MAP = {1: 'Slight', 2: 'Medium', 3: 'Heavy'}

occ_counts = df['occlusion'].value_counts().sort_index()
print('🕶️  Occlusion Level Distribution:')
for occ, cnt in occ_counts.items():
    label = OCCLUSION_MAP.get(occ, str(occ))
    bar   = '█' * int(cnt / occ_counts.max() * 30)
    print(f'   [{occ}] {label:8s}: {cnt:>8,}  ({cnt/len(df)*100:.1f}%)  {bar}')

In [ ]:
# ── Zoom / scale diversity ─────────────────────────────────────────────────────
# 1=no_zoom, 2=zoom_in
ZOOM_MAP = {1: 'No Zoom', 2: 'Zoom In'}
zoom_counts = df['zoom_in'].value_counts().sort_index()
print('🔍  Zoom Diversity:')
for z, cnt in zoom_counts.items():
    print(f'   [{z}] {ZOOM_MAP.get(z, str(z)):12s}: {cnt:>8,}  ({cnt/len(df)*100:.1f}%)')

In [ ]:
# ── Multi-garment images ───────────────────────────────────────────────────────
items_per_image = df.groupby('file')['category_id'].count()

print('👕  Items per Image (garment count per photo):')
print(items_per_image.describe().round(2).to_string())
print()
print(f'   Single-item images   : {(items_per_image == 1).sum():,}')
print(f'   2-item images        : {(items_per_image == 2).sum():,}')
print(f'   3+ item images       : {(items_per_image >= 3).sum():,}')

plt.figure(figsize=(8, 4))
items_per_image.value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='white')
plt.xlabel('Number of Garment Items per Image')
plt.ylabel('Image Count')
plt.title('Multi-Garment Complexity Distribution', fontweight='bold')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ── Source / capture diversity ────────────────────────────────────────────────
source_counts = df['source'].value_counts()
print('📸  Image Source Distribution (shop vs user/consumer):')
for src, cnt in source_counts.items():
    print(f'   {src:30s}: {cnt:>8,}  ({cnt/len(df)*100:.1f}%)')

plt.figure(figsize=(7, 4))
source_counts.plot(kind='bar', color=sns.color_palette('pastel'), edgecolor='white')
plt.title('Image Source / Capture Style Distribution', fontweight='bold')
plt.ylabel('Garment Instance Count')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# ── Diversity overview grid ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. Category imbalance
axes[0, 0].barh(cat_counts['category_name'], cat_counts['count'],
                color=sns.color_palette('tab10', len(cat_counts)))
axes[0, 0].set_title('Category Distribution')
axes[0, 0].set_xlabel('Instances')
axes[0, 0].invert_yaxis()

# 2. Viewpoint
vp_labels = [VIEWPOINT_MAP.get(k, str(k)) for k in viewpoint_counts.index]
axes[0, 1].pie(viewpoint_counts.values, labels=vp_labels, autopct='%1.1f%%',
               colors=['#4C72B0', '#DD8452'], startangle=90)
axes[0, 1].set_title('Viewpoint Distribution')

# 3. Occlusion
occ_labels = [OCCLUSION_MAP.get(k, str(k)) for k in occ_counts.index]
axes[0, 2].bar(occ_labels, occ_counts.values,
               color=['#55A868', '#C44E52', '#8172B2'])
axes[0, 2].set_title('Occlusion Level')
axes[0, 2].set_ylabel('Instances')

# 4. Items per image
ipf = items_per_image.value_counts().sort_index()
axes[1, 0].bar(ipf.index.astype(str), ipf.values, color='mediumseagreen', edgecolor='white')
axes[1, 0].set_title('Items per Image')
axes[1, 0].set_xlabel('Garment Count')

# 5. Source
axes[1, 1].barh(source_counts.index, source_counts.values,
                color=sns.color_palette('pastel', len(source_counts)))
axes[1, 1].set_title('Image Source')
axes[1, 1].set_xlabel('Instances')

# 6. Zoom
zoom_labels = [ZOOM_MAP.get(k, str(k)) for k in zoom_counts.index]
axes[1, 2].pie(zoom_counts.values, labels=zoom_labels, autopct='%1.1f%%',
               colors=['#4C72B0', '#c7e9b4'], startangle=90)
axes[1, 2].set_title('Zoom Diversity')

plt.suptitle('Dataset Diversity Overview', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

---
## Stage 6 — Dataset Selection Plan

We propose a principled strategy for selecting a **15,000–25,000 image subset** that is high-quality, diverse and annotation-complete — without introducing bias toward easy/clean images.

No images are selected or exported yet. This is a plan for review.

In [ ]:
# ── Per-category quotas ──────────────────────────────────────────────────────
#
# Goal: 15,000–25,000 images.
# Strategy: Stratified sampling — each category contributes proportionally,
# but rare categories are OVERSAMPLED with a minimum floor to avoid under-representation.

TARGET_TOTAL     = 20_000   # midpoint target — adjust to taste
MIN_PER_CATEGORY = 300      # floor for rare categories

# Count unique images per category (not garment instances)
cat_img_counts = (
    df.groupby('category_name')['file']
      .nunique()
      .reset_index(name='unique_images')
      .sort_values('unique_images', ascending=False)
)

total_images = cat_img_counts['unique_images'].sum()

# Proportional quota + floor
cat_img_counts['proportional_quota'] = (
    (cat_img_counts['unique_images'] / total_images * TARGET_TOTAL).round().astype(int)
)
cat_img_counts['final_quota'] = cat_img_counts[['proportional_quota',
                                                 pd.Series([MIN_PER_CATEGORY]*len(cat_img_counts))]].max(axis=1)
cat_img_counts['final_quota'] = cat_img_counts.apply(
    lambda r: min(r['final_quota'], r['unique_images']), axis=1
)

cat_img_counts['pct_of_target'] = (cat_img_counts['final_quota'] / TARGET_TOTAL * 100).round(1)
cat_img_counts['available_images'] = cat_img_counts['unique_images']

print(f'📐 Category Sampling Quotas (Target: {TARGET_TOTAL:,} images):')
print(cat_img_counts[['category_name', 'available_images',
                       'proportional_quota', 'final_quota', 'pct_of_target']].to_string(index=False))
print(f'\n   Total planned selection: {cat_img_counts["final_quota"].sum():,} images')

In [ ]:
# ── Selection criteria (scoring rubric) ──────────────────────────────────────
#
# Each image candidate gets a composite score. High scorers are preferred.
# This avoids trivially selecting only the cleanest/easiest images.

selection_criteria = {
    'Annotation Completeness (+)': [
        'Must have at least 1 valid bounding box',
        'Must have segmentation polygon (prefer > 6 points)',
        'Prefer items with landmark annotations',
    ],
    'Quality Gates (hard cutoffs)': [
        f'Remove corrupted / unreadable images ({len(corrupted_files)} flagged)',
        'Remove images with no valid bounding box AND no segmentation',
        'Flag but DO NOT auto-remove blurry images (blur_score < 100) — retain difficult examples intentionally',
    ],
    'Diversity Preservation (+)': [
        'Balance across all 13 garment categories via stratified quota',
        'Maintain occlusion distribution: include Slight / Medium / Heavy proportionally',
        'Maintain viewpoint distribution: No Flip + Flip',
        'Preserve source balance: shop images and consumer/user images',
        'Preserve zoom distribution: standard and zoomed-in crops',
        'Include both single-garment and multi-garment images',
    ],
    'Near-Duplicate Avoidance': [
        'Compute perceptual hash (pHash) fingerprints for all images',
        'Within each category stratum, rank by pHash uniqueness',
        'Discard images with Hamming distance < 5 from an already-selected image',
    ],
    'Challenging Example Retention': [
        'Deliberately retain ≥10% of images with heavy occlusion (occlusion=3)',
        'Deliberately retain ≥10% with blur_score < 100 (real-world imperfections)',
        'Include images with 3+ garments (complex multi-item scenes)',
    ],
    'Annotation Integrity': [
        'Never split an image from its annotation JSON',
        'Preserve original file stems — no renaming',
        'Write selection manifest (CSV) only — do not copy or move images',
    ],
}

print('🗺️  DATASET SELECTION STRATEGY\n')
print(f'   Target subset size : {TARGET_TOTAL:,} images  ({TARGET_TOTAL//1000}k)')
print(f'   Min floor per cat  : {MIN_PER_CATEGORY:,} images\n')
for section, points in selection_criteria.items():
    print(f'  ▶  {section}')
    for p in points:
        print(f'      • {p}')
    print()

In [ ]:
# ── Visualise the planned quota ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Available vs quota
y = np.arange(len(cat_img_counts))
axes[0].barh(y, cat_img_counts['available_images'], height=0.5,
             label='Available', color='lightsteelblue', edgecolor='white')
axes[0].barh(y, cat_img_counts['final_quota'], height=0.5,
             label='Selected Quota', color='steelblue', alpha=0.9, edgecolor='white')
axes[0].set_yticks(y)
axes[0].set_yticklabels(cat_img_counts['category_name'], fontsize=9)
axes[0].set_xlabel('Image Count')
axes[0].set_title('Available Images vs Planned Quota')
axes[0].legend()
axes[0].invert_yaxis()

# Quota pie
axes[1].pie(
    cat_img_counts['final_quota'],
    labels=cat_img_counts['category_name'],
    autopct='%1.1f%%',
    colors=sns.color_palette('tab10', len(cat_img_counts)),
    startangle=140,
    pctdistance=0.75,
    textprops={'fontsize': 7},
)
axes[1].set_title(f'Planned Subset Composition ({TARGET_TOTAL:,} images)')

plt.suptitle('Dataset Selection Plan — Category Quotas', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

---
## Stage 7 — Dataset Split Design

We propose training, validation and testing splits from the **selected subset only** (not from the full dataset).  
Data leakage prevention: images from the same source identity (where identifiable) go into the same split.

In [ ]:
# ── Recommended split ratios ──────────────────────────────────────────────────

SPLIT_RATIOS = {'train': 0.75, 'validation': 0.15, 'test': 0.10}

print('📐 Proposed Split Design')
print(f'   Total subset target : {TARGET_TOTAL:,} images')
print()
split_plan = {}
for split, ratio in SPLIT_RATIOS.items():
    n = round(TARGET_TOTAL * ratio)
    split_plan[split] = n
    print(f'   {split:<12}: {ratio*100:.0f}%  →  {n:>6,} images')

print(f"\n   Total              :        {sum(split_plan.values()):>6,} images")

print("""
Data Leakage Prevention Strategy:
──────────────────────────────────
• DeepFashion2 images in the 'train' directory → candidate pool for Train + Val.
• DeepFashion2 images in the 'validation' directory → reserved for Test split.
• Within the train candidate pool, stratified random split by (category, source)
  ensures class distribution is preserved across Train and Val.
• Generated crops or augmented variants of an image ALWAYS stay
  in the same split as their source image.
• File stems are recorded in a manifest CSV — no images are copied or moved.
"""
)

In [ ]:
# ── Visualise split plan ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar
split_labels = list(split_plan.keys())
split_values = list(split_plan.values())
colors = ['#4C72B0', '#55A868', '#C44E52']
bars = axes[0].bar(split_labels, split_values, color=colors, edgecolor='white')
for bar, v in zip(bars, split_values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{v:,}', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Image Count per Proposed Split')
axes[0].set_ylabel('Images')

# Stacked bar showing split composition per category
cat_names = cat_img_counts['category_name'].tolist()
train_counts = (cat_img_counts['final_quota'] * SPLIT_RATIOS['train']).round().astype(int)
val_counts   = (cat_img_counts['final_quota'] * SPLIT_RATIOS['validation']).round().astype(int)
test_counts  = (cat_img_counts['final_quota'] * SPLIT_RATIOS['test']).round().astype(int)

y = np.arange(len(cat_names))
axes[1].barh(y, train_counts, label='Train',      color='#4C72B0', alpha=0.85)
axes[1].barh(y, val_counts,   left=train_counts,  label='Val',       color='#55A868', alpha=0.85)
axes[1].barh(y, test_counts,  left=train_counts+val_counts, label='Test', color='#C44E52', alpha=0.85)
axes[1].set_yticks(y)
axes[1].set_yticklabels(cat_names, fontsize=8)
axes[1].set_xlabel('Images')
axes[1].set_title('Split Allocation per Category')
axes[1].legend()
axes[1].invert_yaxis()

plt.suptitle('Proposed Train / Val / Test Split Design', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

---
## Final Summary & Audit Report

Compile all findings into a structured JSON audit report saved to `data/metadata/dataset_audit_report.json`.

In [ ]:
# ── Build audit report dict ────────────────────────────────────────────────────

def safe_int(val):
    """Convert numpy int types to Python int for JSON serialisation."""
    return int(val) if hasattr(val, 'item') else val

def safe_float(val):
    return round(float(val), 4) if hasattr(val, 'item') else val

audit_report = {
    'generated_at'   : datetime.now().isoformat(),
    'dataset'        : 'DeepFashion2',
    'dataset_root'   : str(DATASET_ROOT),
    'phase'          : 'Phase 1 — Dataset Exploration',

    'inventory': {
        split: {
            'image_count'       : safe_int(inv['image_count']),
            'anno_count'        : safe_int(inv['anno_count']),
            'matched_pairs'     : safe_int(inv['matched_pairs']),
            'imgs_missing_anno' : safe_int(len(inv['imgs_missing_anno'])),
            'annos_missing_img' : safe_int(len(inv['annos_missing_img'])),
            'format_counts'     : {k: safe_int(v) for k, v in inv['format_counts'].items()},
        }
        for split, inv in inventory.items()
    },

    'annotation_stats': {
        'total_garment_records'  : safe_int(len(df)),
        'with_valid_bbox'        : safe_int(df['has_bbox'].sum()),
        'with_segmentation'      : safe_int(df['has_seg'].sum()),
        'with_landmarks'         : safe_int(df['has_landmarks'].sum()),
        'avg_seg_points'         : safe_float(df['seg_point_count'].mean()),
        'avg_items_per_image'    : safe_float(df.groupby('file').size().mean()),
        'category_distribution'  : {
            row['category_name']: safe_int(row['count'])
            for _, row in cat_counts.iterrows()
        },
        'occlusion_distribution' : {
            OCCLUSION_MAP.get(k, str(k)): safe_int(v)
            for k, v in occ_counts.items()
        },
        'viewpoint_distribution' : {
            VIEWPOINT_MAP.get(k, str(k)): safe_int(v)
            for k, v in viewpoint_counts.items()
        },
        'source_distribution'    : {
            str(k): safe_int(v) for k, v in source_counts.items()
        },
    },

    'image_quality': {
        'sample_size'            : safe_int(len(df_quality)),
        'corrupted_count'        : safe_int(len(corrupted_files)),
        'corrupted_files'        : [c['file'] for c in corrupted_files[:50]],
        'blurry_count'           : safe_int((df_quality['blur_score'] < BLUR_THRESHOLD).sum()),
        'blur_threshold'         : BLUR_THRESHOLD,
        'median_width_px'        : safe_float(df_quality['width'].median()),
        'median_height_px'       : safe_float(df_quality['height'].median()),
        'median_aspect_ratio'    : safe_float(df_quality['aspect'].median()),
        'median_file_size_kb'    : safe_float(df_quality['file_size_kb'].median()),
        'median_blur_score'      : safe_float(df_quality['blur_score'].median()),
        'median_brightness'      : safe_float(df_quality['brightness'].median()),
        'median_contrast'        : safe_float(df_quality['contrast'].median()),
    },

    'selection_plan': {
        'target_subset_size'    : TARGET_TOTAL,
        'min_per_category'      : MIN_PER_CATEGORY,
        'strategy_summary'      : [
            'Stratified sampling by category with minimum floor for rare classes',
            'Hard-filter corrupted images and those with no annotations',
            'Retain blurry and occluded images for challenge diversity',
            'Near-duplicate removal via perceptual hashing (Phase 2)',
            'Balance source distribution (shop vs consumer)',
        ],
        'category_quotas': {
            row['category_name']: safe_int(row['final_quota'])
            for _, row in cat_img_counts.iterrows()
        },
    },

    'split_plan': {
        'ratios'  : SPLIT_RATIOS,
        'counts'  : {k: safe_int(v) for k, v in split_plan.items()},
        'leakage_prevention': [
            'DeepFashion2 train dir → Train + Val candidate pool',
            'DeepFashion2 validation dir → Test split only',
            'Augmented crops always stay in same split as source image',
            'Splits recorded as file-stem manifests (CSV) — no image movement',
        ],
    },

    'recommendations': [
        'High class imbalance detected — apply category-stratified sampling with floors.',
        'Segmentation coverage should be verified per-category before training.',
        'Corrupted images are catalogued — remove from selection manifest only, not from disk.',
        'Blurry images should be retained to prevent training bias toward clean studio shots.',
        'Near-duplicate detection (pHash) is the critical next step in preprocessing.',
        'Consider oversampling rare categories (vest, sling, sling dress) in Phase 2.',
    ],
}

with open(AUDIT_REPORT, 'w') as f:
    json.dump(audit_report, f, indent=2)

print(f'✅ Audit report saved → {AUDIT_REPORT}')

In [ ]:
# ── Print final human-readable summary ───────────────────────────────────────
print('=' * 65)
print('  FITLOOP — PHASE 1 AUDIT SUMMARY')
print('=' * 65)

total_imgs  = sum(inv['image_count']  for inv in inventory.values())
total_annos = sum(inv['anno_count']   for inv in inventory.values())
total_match = sum(inv['matched_pairs'] for inv in inventory.values())

print(f'\n📦  DATASET SIZE')
print(f'    Total images          : {total_imgs:>10,}')
print(f'    Total annotations     : {total_annos:>10,}')
print(f'    Matched pairs         : {total_match:>10,}')
print(f'    Total garment records : {len(df):>10,}')
print(f'    Categories            : {df["category_name"].nunique():>10}')

print(f'\n📝  ANNOTATION COMPLETENESS')
print(f'    Valid bounding boxes  : {df["has_bbox"].mean()*100:>9.1f}%')
print(f'    Segmentation polygons : {df["has_seg"].mean()*100:>9.1f}%')
print(f'    Landmark annotations  : {df["has_landmarks"].mean()*100:>9.1f}%')

print(f'\n🏥  IMAGE QUALITY')
print(f'    Corrupted images      : {len(corrupted_files):>10,}  (of {len(sample_img_paths):,} sampled)')
print(f'    Blurry (score < {BLUR_THRESHOLD:.0f}) : '
      f'{(df_quality["blur_score"] < BLUR_THRESHOLD).sum():>10,}  ({(df_quality["blur_score"] < BLUR_THRESHOLD).mean()*100:.1f}%)')
print(f'    Median resolution     : '
      f'{int(df_quality["width"].median())} × {int(df_quality["height"].median())} px')

print(f'\n⚖️   CLASS IMBALANCE')
max_cat  = cat_counts.iloc[0]
min_cat  = cat_counts.iloc[-1]
imb_ratio = max_cat['count'] / min_cat['count']
print(f'    Most common  : {max_cat["category_name"]} ({max_cat["count"]:,})')
print(f'    Rarest       : {min_cat["category_name"]} ({min_cat["count"]:,})')
print(f'    Imbalance ratio : {imb_ratio:.1f}×')
if imb_ratio > 10:
    print('    ⚠️  Severe imbalance — stratified sampling with floor is essential.')
elif imb_ratio > 5:
    print('    ⚠️  Moderate imbalance — stratified sampling recommended.')
else:
    print('    ✅  Class distribution is relatively balanced.')

print(f'\n🗂️   SELECTION PLAN')
print(f'    Target subset        : {TARGET_TOTAL:>10,} images')
print(f'    Train split          : {split_plan["train"]:>10,} images ({SPLIT_RATIOS["train"]*100:.0f}%)')
print(f'    Validation split     : {split_plan["validation"]:>10,} images ({SPLIT_RATIOS["validation"]*100:.0f}%)')
print(f'    Test split           : {split_plan["test"]:>10,} images ({SPLIT_RATIOS["test"]*100:.0f}%)')

print(f'\n💡  RECOMMENDATIONS FOR PHASE 2')
for i, rec in enumerate(audit_report['recommendations'], 1):
    print(f'    {i}. {rec}')

print(f'\n📄  Full audit saved → {AUDIT_REPORT}')
print('=' * 65)
print('  ✅  Phase 1 Exploration Complete. Ready for Phase 2 Review.')
print('=' * 65)